# QL Baseline Runner

This notebook launches fresh Q-learning baselines on `matlab_env_python_replica` and renders the saved baseline plots inline.

Default scenario for this notebook:
- `30 s` episode duration
- `10 s` skin-to-fat switch
- midpoint reset to keep the piston away from the physical hard end at episode start
- stroke limit handled as a clamped end-stop, not episode death
- force input `5 N` amplitude, `15 N` bias, `5 rad/s`
- finer Q-learning action set `[-2, -1.5, -1, -0.5, 0, 0.5, 1, 1.5, 2] V`
- both FE modes: `switched_dynamics` and `gui_skin_locked`


In [ ]:
import subprocess
import sys
from pathlib import Path

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    for root in (candidate, candidate / 'TeleopWithRL'):
        if (root / 'matlab_env_python_replica').exists() and (root / 'notebooks' / '_teleop_nb.py').exists():
            if str(root) not in sys.path:
                sys.path.insert(0, str(root))
            break
    else:
        continue
    break
else:
    raise RuntimeError('Could not find TeleopWithRL notebook root.')

from notebooks._teleop_nb import load_json, repo_paths, show_image, show_markdown, show_rows

P = repo_paths()
REPO = P['repo']
WORKSPACE = REPO.parent
QL_RESULTS = P['ql_results']

CFG = {
    'study_name': 'ql_30s10s_mid_5A15B_w5_a05_clamp_te1k_01',
    'env_mode': 'changing_skin_fat',
    'episode_duration_s': 30.0,
    'env_switch_time_s': 10.0,
    'reset_position_mode': 'midpoint',
    'stroke_limit_mode': 'clamp',
    'force_amp_N': 5.0,
    'force_bias_N': 15.0,
    'force_freq_rad_s': 5.0,
    'force_phase_rad': 0.0,
    'action_levels_V': [-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0],
    'force_waveform': 'sine',
    'q_episodes': 2000,
    'test_episodes': 300,
    'seed': 42,
    'parallel_workers': 1,
    'worker_torch_threads': 1,
    'skip_existing': True,
    'resume': False,
    'run_parallel': False,
    'disable_stroke_limit': False,
}

MODE_LABELS = {
    'dyn': 'switched_dynamics',
    'gui': 'gui_skin_locked',
}
RUN_ROOTS = {
    mode: QL_RESULTS / mode / f"{CFG['study_name']}_{mode}" / '00b' / 'ql'
    for mode in ('dyn', 'gui')
}

CMD = [
    sys.executable,
    '-m',
    'TeleopWithRL.matlab_env_python_replica.ql_experiments.run_ql_baselines_both_fe',
    '--study-name', CFG['study_name'],
    '--env-mode', CFG['env_mode'],
    '--episode-duration', str(CFG['episode_duration_s']),
    '--env-switch-time', str(CFG['env_switch_time_s']),
    '--reset-position-mode', CFG['reset_position_mode'],
    '--stroke-limit-mode', CFG['stroke_limit_mode'],
    '--force-amp', str(CFG['force_amp_N']),
    '--force-bias', str(CFG['force_bias_N']),
    '--force-freq-rad', str(CFG['force_freq_rad_s']),
    '--force-phase', str(CFG['force_phase_rad']),
    '--force-waveform', CFG['force_waveform'],
    '--action-levels', *[str(v) for v in CFG['action_levels_V']],
    '--q-episodes', str(CFG['q_episodes']),
    '--test-episodes', str(CFG['test_episodes']),
    '--seed', str(CFG['seed']),
    '--parallel-workers', str(CFG['parallel_workers']),
    '--worker-torch-threads', str(CFG['worker_torch_threads']),
]
if CFG['skip_existing']:
    CMD.append('--skip-existing')
if CFG['resume']:
    CMD.append('--resume')
if CFG['run_parallel']:
    CMD.append('--run-parallel')
if CFG['disable_stroke_limit']:
    CMD.append('--disable-stroke-limit')

show_rows(
    [
        {'item': 'workspace_root', 'value': str(WORKSPACE)},
        {'item': 'study_name', 'value': CFG['study_name']},
        {'item': 'episode_duration_s', 'value': CFG['episode_duration_s']},
        {'item': 'env_switch_time_s', 'value': CFG['env_switch_time_s']},
        {'item': 'reset_position_mode', 'value': CFG['reset_position_mode']},
        {'item': 'stroke_limit_mode', 'value': CFG['stroke_limit_mode']},
        {'item': 'force_amp_N', 'value': CFG['force_amp_N']},
        {'item': 'force_bias_N', 'value': CFG['force_bias_N']},
        {'item': 'force_freq_rad_s', 'value': CFG['force_freq_rad_s']},
        {'item': 'action_levels_V', 'value': CFG['action_levels_V']},
        {'item': 'q_episodes', 'value': CFG['q_episodes']},
        {'item': 'test_episodes', 'value': CFG['test_episodes']},
        {'item': 'disable_stroke_limit', 'value': CFG['disable_stroke_limit']},
        {'item': 'dyn_run_root', 'value': str(RUN_ROOTS['dyn'])},
        {'item': 'gui_run_root', 'value': str(RUN_ROOTS['gui'])},
        {'item': 'command', 'value': subprocess.list2cmdline(CMD)},
    ],
    title='QL baseline run config',
    max_rows=20,
)


In [ ]:
print(subprocess.list2cmdline(CMD))
completed = subprocess.run(CMD, cwd=str(WORKSPACE), check=True)
print(f'Completed with return code {completed.returncode}.')


In [ ]:
summary_rows = []
artifact_rows = []
for mode, run_root in RUN_ROOTS.items():
    summary_path = run_root / 'l' / 'summary.json'
    plots_dir = run_root / 'p'
    artifact_rows.append({'fe_mode': MODE_LABELS[mode], 'summary_json': str(summary_path), 'plots_dir': str(plots_dir)})
    if not summary_path.exists():
        summary_rows.append({'fe_mode': MODE_LABELS[mode], 'status': 'missing', 'run_root': str(run_root)})
        continue
    data = load_json(summary_path)
    reset_options = dict(data.get('reset_options', {}))
    summary_rows.append({
        'fe_mode': MODE_LABELS[mode],
        'label': data.get('label'),
        'tracking_rmse_m': data.get('tracking_rmse_m'),
        'transparency_rmse_w': data.get('transparency_rmse_w'),
        'pre_switch_tracking_rmse_m': data.get('pre_switch_tracking_rmse_m'),
        'post_switch_tracking_rmse_m': data.get('post_switch_tracking_rmse_m'),
        'pre_switch_transparency_rmse_w': data.get('pre_switch_transparency_rmse_w'),
        'post_switch_transparency_rmse_w': data.get('post_switch_transparency_rmse_w'),
        'invalid_episode_rate': data.get('invalid_episode_rate'),
        'evaluation_episodes': data.get('evaluation_episodes'),
        'completed_episodes': data.get('completed_episodes'),
        'completed_episode_rate': data.get('completed_episode_rate'),
        'terminated_episodes': data.get('terminated_episodes'),
        'stroke_limit_episodes': data.get('stroke_limit_episodes'),
        'volume_singularity_episodes': data.get('volume_singularity_episodes'),
        'tracking_error_fail_episodes': data.get('tracking_error_fail_episodes'),
        'mean_episode_seconds': data.get('mean_episode_seconds'),
        'episode_duration': data.get('episode_duration'),
        'env_switch_time': data.get('env_switch_time'),
        'stroke_limit_mode': data.get('stroke_limit_mode'),
        'force_amp': reset_options.get('force_amp'),
        'force_bias': reset_options.get('force_bias'),
        'force_freq_rad': reset_options.get('force_freq_rad'),
        'action_levels': data.get('action_levels'),
        'reset_position_mode': reset_options.get('reset_position_mode'),
        'enforce_stroke_limit': data.get('enforce_stroke_limit'),
        'stroke_stop_hit_episodes': data.get('stroke_stop_hit_episodes'),
        'state_variant': data.get('state_variant'),
        'reward_variant': data.get('reward_variant'),
        'out_dir': data.get('out_dir'),
    })

show_rows(summary_rows, title='QL baseline summaries', max_rows=10)
show_rows(artifact_rows, title='QL artifact locations', max_rows=10)


In [ ]:
plot_specs = [
    ('train.png', 'Training metrics'),
    ('avg_roll.png', 'Average evaluation rollout'),
    ('roll.png', 'Evaluation roll plot'),
    ('act.png', 'Evaluation action plot'),
    ('traj.png', 'Evaluation trajectory plot'),
    ('polmap.png', 'Policy map'),
    ('visit.png', 'State visit heatmap'),
]

for mode, run_root in RUN_ROOTS.items():
    show_markdown(f"## {MODE_LABELS[mode]}")
    for filename, title in plot_specs:
        show_image(run_root / 'p' / filename, title=f"{MODE_LABELS[mode]}: {title}")
